# CMU 10-799 — Colab training
Runtime → Change runtime type → **GPU** (A100 or L4 with Colab Pro) before running.

Workflow: edit code locally → `git push` to your fork → re-run **§1** here → train.
All checkpoints/samples/logs go to Drive under `MyDrive/cmu-10799/logs/`, so a disconnect only costs progress since the last `save_every`.

## 0. GPU check + mount Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cmu-10799'
import os; os.makedirs(f'{DRIVE}/logs', exist_ok=True); os.makedirs(f'{DRIVE}/data', exist_ok=True)

## 1. Pull code from your fork
Re-run this cell any time you push new code.

In [ ]:
%cd /content
REPO = 'https://github.com/avnithv/cmu-10799-diffusion.git'
if os.path.isdir('cmu-10799-diffusion'):
    %cd cmu-10799-diffusion
    !git pull
else:
    !git clone $REPO
    %cd cmu-10799-diffusion
!git log --oneline -1

## 2. Install extra dependencies
Colab already has torch/torchvision/numpy/pillow/pyyaml/tqdm/scipy/matplotlib.

In [ ]:
!pip install -q einops torch-fidelity datasets huggingface-hub wandb
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Dataset
First run: downloads from HuggingFace and saves in Arrow format to Drive (one-time).
Every run: copies the Arrow files from Drive to local disk (`/content/data/celeba`) — training reads local disk, which is much faster than Drive.

In [ ]:
DRIVE_DATA = f'{DRIVE}/data/celeba'
LOCAL_DATA = '/content/data/celeba'   # must match data.root in configs/ddpm_colab.yaml

if not os.path.exists(f'{DRIVE_DATA}/dataset_dict.json'):
    from datasets import load_dataset
    ds = load_dataset('electronickale/cmu-10799-celeba64-subset')
    ds.save_to_disk(DRIVE_DATA)
    print('Saved dataset to Drive:', DRIVE_DATA)

if not os.path.exists(f'{LOCAL_DATA}/dataset_dict.json'):
    !mkdir -p /content/data && cp -r "$DRIVE_DATA" /content/data/
print('Local dataset ready:', os.listdir(LOCAL_DATA))

## 4. (Optional) Weights & Biases
Skip this and set `logging.wandb.enabled: false` in the config if you don't want W&B.

In [ ]:
import wandb; wandb.login()

## 5. Train
Quick sanity check first (overfits one batch), then the real run.

In [ ]:
# Sanity check: loss should drop fast
!python train.py --method ddpm --config configs/ddpm_colab.yaml --overfit-single-batch

In [ ]:
!python train.py --method ddpm --config configs/ddpm_colab.yaml

## 6. Resume after a disconnect
Lists checkpoints on Drive; set `CKPT` to the one you want.

In [ ]:
!find $DRIVE/logs -name '*.pt' | sort
CKPT = ''   # e.g. f'{DRIVE}/logs/ddpm_20260901_120000/checkpoints/ddpm_0005000.pt'
if CKPT:
    !python train.py --method ddpm --config configs/ddpm_colab.yaml --resume "$CKPT"

## 7. Sample + evaluate (FID via torch-fidelity)

In [ ]:
CKPT = ''   # final checkpoint path on Drive
if CKPT:
    !python sample.py --method ddpm --checkpoint "$CKPT" --num_samples 64 --grid --output_dir $DRIVE/samples
    # see scripts/evaluate_torch_fidelity.sh for options (--dataset_path, --num_samples, --metrics)
    !bash scripts/evaluate_torch_fidelity.sh ddpm "$CKPT" --dataset-path /content/data/celeba